## reshape ו-transpose

שני כלים לשינוי **צורת** מערך בלי לשנות את הערכים עצמם:

- **‎`a.reshape(new_shape)`‎** — מסדר מחדש את אותם ערכים לתוך צורה חדשה, בסדר קריאה קבוע (שורה-שורה). לא "מזיז" נתונים בין ריצות/זמנים — רק קורא להם אחרת.
- **‎`a.T`‎** (או ‎`a.transpose()`‎) — **מחליף בין הצירים עצמם**: ‎`(5, 60)`‎ הופך ל-‎`(60, 5)`‎. בניגוד ל-reshape, זו לא רק "קריאה אחרת" — ציר 0 וציר 1 מחליפים תפקידים.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0_nom = 20.0                       # nominal launch speed (m/s)
# measured launch speed of each run (m/s):
v0_runs = np.array([18.0, 19.5, 20.0, 20.5, 22.0])
n_runs, n_t = 5, 60
t_flight_nom = 2 * v0_nom * np.sin(theta) / g
t = np.linspace(0, t_flight_nom, n_t)   # the same time grid for all runs

Y = np.zeros((n_runs, n_t))
for i in range(n_runs):
    Y[i] = v0_runs[i] * np.sin(theta) * t - 0.5 * g * t**2

### דוגמה: transpose לצורך תאימות לכלי חיצוני

נניח שכלי חיצוני (למשל טבלת pandas שנפגוש בשבוע 10) מצפה לקבל **שורה לכל נקודת זמן ועמודה לכל ריצה** — בדיוק ההפך מ-‎`Y`‎.

In [ ]:
Y_time_major = Y.T
print("Y.shape:", Y.shape, "  Y_time_major.shape:", Y_time_major.shape)

# check: the value did not change, only its place
print(Y[2, 10] == Y_time_major[10, 2])

### דוגמה: reshape לצורך שטוח (flatten)

לפעמים רוצים את כל 300 המדידות ($5\times 60$) כרשימה שטוחה אחת — למשל לחישוב היסטוגרמה כללית על כל המדידות ביחד (סעיף 8.8), בלי חלוקה לריצות.

In [ ]:
Y_flat = Y.reshape(-1)        # -1 means "work out this size yourself"
print(Y_flat.shape)           # (300,)

# and back to the original shape:
Y_back = Y_flat.reshape(n_runs, n_t)
print(np.all(Y == Y_back))

### באג נפוץ: ‎`reshape`‎ בהנחה שגויה על סדר הנתונים

‎`reshape`‎ **לא** מודע למשמעות הפיזיקלית של הצירים — הוא רק קורא את הזיכרון בסדר קבוע ומחלק אותו לצורה החדשה. אם משנים ‎`shape`‎ בלי לוודא שהמשמעות עדיין נכונה (למשל, ‎`reshape(n_t, n_runs)`‎ על ‎`Y`‎ שצורתו ‎`(n_runs, n_t)`‎, במקום ‎`Y.T`‎), מקבלים מערך **באותה צורה שרצינו**, אבל עם ערכים **מעורבבים בין ריצות וזמנים** — שגיאה שקטה לגמרי.

In [ ]:
wrong = Y.reshape(n_t, n_runs)     # no error, but the values are mixed up
right = Y.T                        # the right way to swap the axes

print(np.all(wrong == right))      # False: this is the bug

### נסו בעצמכם

בתא הבא יש שתי ריצות קצרות, עם 3 נקודות זמן בכל אחת.

**לפני שמריצים:** כתבו מה תדפיס כל אחת משתי שורות ההדפסה. בשתיהן מתקבל מערך בצורה ‎`(3, 2)`‎. באיזו מהן כל שורה היא נקודת זמן אחת, עם הגובה של שתי הריצות?

In [ ]:
Y2 = np.array([[0.0, 1.5, 2.0],
               [0.0, 2.5, 4.0]])   # 2 runs, 3 times: (run, time)
print(Y2.T)
print(Y2.reshape(3, 2))

`````{admonition} פתרון
:class: dropdown, tip
```text
[[0.  0. ]
 [1.5 2.5]
 [2.  4. ]]
[[0.  1.5]
 [2.  0. ]
 [2.5 4. ]]
```
‎`Y2.T`‎ הוא המערך שבו כל שורה היא נקודת זמן אחת: בשורה האמצעית עומדים 1.5 ו-2.5, הגבהים של שתי הריצות באותו זמן.

‎`reshape`‎ קורא את ששת הערכים לפי הסדר (קודם כל הזמנים של ריצה 0, ואחריהם של ריצה 1) ומחלק אותם לזוגות. לכן בשורה האמצעית שלו עומדים 2.0, הגובה האחרון של ריצה 0, ו-0.0, הגובה הראשון של ריצה 1: ערכים משתי ריצות שונות ומשני זמנים שונים.
`````

### בדקו את עצמכם

השאלה השנייה מתייחסת לקוד הבא:

```python
a = np.array([[1, 2, 3],
              [4, 5, 6]])
print(a.T[0])
```

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מה ההבדל המהותי בין reshape לבין T?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "reshape משנה את הערכים עצמם, T לא",
                "correct": false,
                "feedback": "לא — אף אחת מהן לא משנה ערכים, רק את סידורם."
            },
            {
                "answer": "הן פונקציות זהות לחלוטין",
                "correct": false,
                "feedback": "לא נכון — ראינו דוגמה שבה הן נותנות תוצאות שונות."
            },
            {
                "answer": "T תמיד יוצר עותק, ו-reshape תמיד יוצר תצוגה",
                "correct": false,
                "feedback": "לא — שתי הפעולות יכולות להחזיר תצוגה או עותק תלוי במקרה; ההבדל המהותי הוא במה שהן עושות לצירים, לא בזיכרון."
            },
            {
                "answer": "T מחליף את תפקידי הצירים תוך שמירת ההתאמה הנכונה; reshape רק קורא מחדש את הזיכרון בסדר קבוע",
                "correct": true,
                "feedback": "נכון."
            }
        ]
    },
    {
        "question": "מהו המספר השני שתדפיס שורת ההדפסה?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "4",
                "correct": true,
                "feedback": "נכון: השורה הראשונה של a.T היא העמודה הראשונה של a, כלומר 1 ו-4."
            },
            {
                "answer": "2",
                "correct": false,
                "feedback": "לא: 1 ו-2 הם תחילת השורה הראשונה של a לפני ההחלפה. T מחליף שורות ועמודות."
            },
            {
                "answer": "5",
                "correct": false,
                "feedback": "לא: 5 נמצא בעמודה האמצעית של a, והיא השורה השנייה של a.T."
            },
            {
                "answer": "3",
                "correct": false,
                "feedback": "לא: a.T מתחיל בעמודה הראשונה של a, 1 ו-4."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)

``````{admonition} תרגול נוסף
:class: dropdown, note
עבור ‎`Y2`‎ מ"נסו בעצמכם": מה תדפיס השורה בתא שלמטה? באיזה אינדקס של המערך השטוח נמצא הגובה של הריצה השנייה בנקודת הזמן השנייה (2.5)?

```python
print(Y2.reshape(-1))
```

`````{admonition} פתרון
:class: dropdown, tip
```text
[0.  1.5 2.  0.  2.5 4. ]
```
המערך השטוח עובר על הריצות לפי הסדר: קודם שלושת הזמנים של ריצה 0 (אינדקסים 0 עד 2), ואחריהם שלושת הזמנים של ריצה 1. הגובה 2.5 הוא הזמן השני של ריצה 1, ולכן הוא באינדקס 4.
`````
``````